## XGBoost

O [XGBoost](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html) (Extreme Gradient Boosting) é um modelo baseado em árvores que utiliza a técnica de boosting, onde múltiplos modelos fracos são combinados sequencialmente para formar um modelo forte.

A cada nova árvore, o modelo tenta corrigir os erros das árvores anteriores, melhorando progressivamente o desempenho.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores no modelo
  - Mais árvores → melhor aprendizado (até certo limite)

- **learning_rate**: taxa de aprendizado
  - Valores menores tornam o aprendizado mais lento, porém mais robusto

- **max_depth**: profundidade das árvores
  - Controla a complexidade do modelo

### Estratégia

Foi utilizado GridSearchCV com validação cruzada (5 folds) e métrica ROC AUC para encontrar a melhor combinação de hiperparâmetros.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from preprocessing import load_and_preprocess

In [14]:
X_train, X_test, y_train, y_test = load_and_preprocess("../predictive_models/scrdata_202505.csv")

In [ ]:
param_grid_xgb = {
    "n_estimators": [200, 400],
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 1],
    "colsample_bytree": [0.7, 1],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [1, 5]
}

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight
    ),
    param_grid=param_grid_xgb,
    cv=5,
    scoring="roc_auc",
    n_jobs=2,
    verbose=1
)

grid_xgb.fit(X_train, y_train)

print("Melhores parâmetros:", grid_xgb.best_params_)
print("Melhor score (CV):", grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_

y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:, 1]

roc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

resultados = {
    "modelo": "XGBoost",
    "roc_auc": roc,
    "f1": f1,
    "accuracy": acc,
    "roc_auc_cv": grid_xgb.best_score_,
    "best_params": str(grid_xgb.best_params_)
}

df_result = pd.DataFrame([resultados])
df_result.to_csv("resultados_xgb.csv", index=False)

Fitting 5 folds for each of 192 candidates, totalling 960 fits
Melhores parâmetros: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 400, 'reg_alpha': 0, 'reg_lambda': 5, 'subsample': 1}
Melhor score: 0.766422788685171
